In [0]:
%pip install openai tiktoken
dbutils.library.restartPython()

In [0]:
import math
import time
import random
import unicodedata
from typing import List

import tiktoken
from openai import AzureOpenAI
from pyspark.sql import functions as F
from pyspark.sql import types as T

# -----------------------------
# CONFIG
# -----------------------------
AZURE_ENDPOINT = "https://mirei-mfr10ulo-canadaeast.cognitiveservices.azure.com/"
DEPLOYMENT_NAME = "text-embedding-3-large-3"
API_VERSION = "2024-12-01-preview"
API_KEY = "key"

OCR_TABLE = "noc_documents_azure"
TEXT_COL = "final_text"
EMBED_COL = "embedding"

MAX_RETRIES = 6
BASE_SLEEP_SEC = 1.5

MODEL_FOR_TOKENS = "text-embedding-3-large"
MAX_TOKENS = 1200
OVERLAP_RATIO = 0.15

# -----------------------------
# CLIENT + TOKENIZER 
# -----------------------------
client = AzureOpenAI(
    azure_endpoint=AZURE_ENDPOINT,
    api_key=API_KEY,
    api_version=API_VERSION
)

enc = tiktoken.encoding_for_model(MODEL_FOR_TOKENS)

# -----------------------------
# TEXT CLEANING
# -----------------------------
def clean_text(text: str) -> str:
    if not text:
        return ""
    text = text.encode("utf-8", "ignore").decode("utf-8", "ignore")
    text = unicodedata.normalize("NFC", text)
    return text.strip()

# -----------------------------
# TOKEN CHUNKING WITH OVERLAP
# -----------------------------
def chunk_by_tokens(
    text: str,
    max_tokens: int = MAX_TOKENS,
    overlap_ratio: float = OVERLAP_RATIO
) -> List[str]:

    tokens = enc.encode(text)
    if not tokens:
        return []

    overlap_tokens = int(max_tokens * overlap_ratio)
    overlap_tokens = max(0, min(overlap_tokens, max_tokens - 1))

    chunks = []
    start = 0
    total_tokens = len(tokens)

    while start < total_tokens:
        end = min(start + max_tokens, total_tokens)
        chunk_tokens = tokens[start:end]
        chunks.append(enc.decode(chunk_tokens))

        if end == total_tokens:
            break

        start = end - overlap_tokens

    return chunks

# -----------------------------
# EMBEDDING WITH RETRIES
# -----------------------------
def embed_batch(texts: List[str]) -> List[List[float]]:
    for attempt in range(MAX_RETRIES):
        try:
            response = client.embeddings.create(
                model=DEPLOYMENT_NAME,
                input=texts
            )

            sorted_data = sorted(response.data, key=lambda x: x.index)
            return [d.embedding for d in sorted_data]

        except Exception as e:
            if attempt == MAX_RETRIES - 1:
                raise
            sleep_s = BASE_SLEEP_SEC * (2 ** attempt) + random.random()
            print(f"Retry {attempt+1}/{MAX_RETRIES} after error: {e}")
            time.sleep(sleep_s)

# -----------------------------
# POOLING + NORMALIZATION
# -----------------------------
def mean_pool(vectors: List[List[float]]) -> List[float]:
    dim = len(vectors[0])
    pooled = [0.0] * dim

    for v in vectors:
        for i in range(dim):
            pooled[i] += float(v[i])

    n = float(len(vectors))
    return [x / n for x in pooled]

def l2_normalize(v: List[float]) -> List[float]:
    norm = math.sqrt(sum(x * x for x in v))
    if norm == 0:
        return v
    return [x / norm for x in v]

# -----------------------------
# CORE PROCESSING FUNCTION
# -----------------------------
def process_documents(limit: int = None):
    """
    Core engine.
    If limit is None → process entire table.
    If limit is set → process limited number of rows.
    Fully rerun-safe.
    """

    try:
        spark.sql(f"ALTER TABLE {OCR_TABLE} ADD COLUMNS ({EMBED_COL} ARRAY<DOUBLE>)")
    except Exception:
        pass

    df = (
        spark.table(OCR_TABLE)
             .select("path", TEXT_COL)
             .where(F.col(TEXT_COL).isNotNull())
             .where(F.col(EMBED_COL).isNull())
    )

    if limit:
        df = df.limit(limit)

    rows = df.collect()
    print("Documents to process:", len(rows))

    for r in rows:
        path = r["path"]
        text = clean_text(r[TEXT_COL])

        try:
            chunks = chunk_by_tokens(text)

            if not chunks:
                print(f"No chunks for: {path}")
                continue

            chunk_vectors = embed_batch(chunks)
            doc_vector = l2_normalize(mean_pool(chunk_vectors))

            update_df = spark.createDataFrame(
                [(path, [float(x) for x in doc_vector])],
                schema=T.StructType([
                    T.StructField("path", T.StringType(), False),
                    T.StructField("embedding", T.ArrayType(T.DoubleType()), True)
                ])
            )

            update_df.createOrReplaceTempView("emb_update_single")

            spark.sql(f"""
            MERGE INTO {OCR_TABLE} t
            USING emb_update_single s
            ON t.path = s.path
            WHEN MATCHED THEN UPDATE SET
              t.{EMBED_COL} = s.embedding
            """)

            print(f"Saved embedding for: {path}")

        except Exception as e:
            print(f"Failed on {path}: {e}")
            continue

    print("Run completed safely.")

# -----------------------------
# ORCHESTRATORS
# -----------------------------
def run_embeddings_full_table():
    """
    Processes entire table (only rows with NULL embedding).
    """
    process_documents(limit=None)

def run_embeddings_limited(row_limit: int):
    """
    Processes limited number of rows (only rows with NULL embedding).
    """
    process_documents(limit=row_limit)


In [0]:
import time
st= time.time()
run_embeddings_limited(100)
et=time.time()
print(et-st)

In [0]:
display(
    spark.table("noc_documents_azure")
         .where("embedding IS NOT NULL")
)

In [0]:
import time
st= time.time()
run_embeddings_full_table()
et=time.time()
print(et-st)